<a href="https://colab.research.google.com/github/chughtaianosh-boop/evolvix-aiml-internship-anoshchughtai/blob/main/Week_2/Week_2_Task_2_Anosh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task 2: Baseline Model, Feature Engineering, and Hyperparameter Tuning on Mushrooms Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# 1. Load Correct Dataset
dataset_path = '/content/dataset/mushrooms.csv'

# If the directory doesn't exist, we'll create it and download the dataset
# (Assuming the dataset might need to be downloaded or already exists there)
if not os.path.exists(dataset_path):
    os.makedirs(os.path.dirname(dataset_path), exist_ok=True)
    # Downloading the dataset from a standard repository to ensure the code executes successfully
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data'
    columns = [
        'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
        'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
        'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
        'stalk-surface-below-ring', 'stalk-color-above-ring',
        'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
        'ring-type', 'spore-print-color', 'population', 'habitat'
    ]
    df = pd.read_csv(url, header=None, names=columns)
    df.to_csv(dataset_path, index=False)
else:
    df = pd.read_csv(dataset_path)

print(f"Dataset loaded successfully. Shape: {df.shape}")
display(df.head())


Dataset loaded successfully. Shape: (8124, 23)


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [ ]:
# 2. Preprocessing
# Label Encoding for the target column ('class')
le = LabelEncoder()
y = le.fit_transform(df['class'])

# Drop target to isolate features
X_raw = df.drop(columns=['class'])

# One-Hot Encoding for categorical features
X_baseline = pd.get_dummies(X_raw, drop_first=True)

# 3. Baseline Model
X_train, X_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)

baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)
baseline_f1 = f1_score(y_test, baseline_preds, average='binary')

print("=== Baseline Model Evaluation ===")
print(f"Accuracy: {baseline_acc * 100:.2f}%")
print(f"F1-Score: {baseline_f1:.4f}")


=== Baseline Model Evaluation ===
Accuracy: 100.00%
F1-Score: 1.0000


In [ ]:
# 4. Feature Engineering
# We will engineer 2 new interaction features:
# Feature 1: Combination of cap-shape and cap-color
# Feature 2: Combination of odor and spore-print-color
df_fe = df.copy()
df_fe['cap_shape_color'] = df_fe['cap-shape'].astype(str) + "_" + df_fe['cap-color'].astype(str)
df_fe['odor_spore_print'] = df_fe['odor'].astype(str) + "_" + df_fe['spore-print-color'].astype(str)

# Drop target column
X_fe_raw = df_fe.drop(columns=['class'])

# One-Hot Encode all categorical variables, including the newly engineered features
X_engineered = pd.get_dummies(X_fe_raw, drop_first=True)

print(f"Features after engineering and encoding: {X_engineered.shape[1]} columns (originally {X_baseline.shape[1]})")


Features after engineering and encoding: 154 columns (originally 95)


In [ ]:
# 5. Hyperparameter Tuning using GridSearchCV on Engineered Dataset
X_train_e, X_test_e, y_train, y_test = train_test_split(X_engineered, y, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)
grid_search.fit(X_train_e, y_train)

best_model = grid_search.best_estimator_
print("Best parameters found:", grid_search.best_params_)

tuned_preds = best_model.predict(X_test_e)
tuned_acc = accuracy_score(y_test, tuned_preds)
tuned_f1 = f1_score(y_test, tuned_preds, average='binary')
print(f"Tuned Model Accuracy: {tuned_acc:.4f}")
print(f"Tuned Model F1-Score: {tuned_f1:.4f}")


Best parameters found: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 50}
Tuned Model Accuracy: 1.0000
Tuned Model F1-Score: 1.0000


In [ ]:
# 6. Performance Comparison
improvement = tuned_acc - baseline_acc
print("=== Performance Comparison ===")
print(f"Baseline Model Accuracy: {baseline_acc * 100:.2f}%")
print(f"Tuned Model Accuracy:    {tuned_acc * 100:.2f}%")
print(f"Absolute Improvement:    {improvement * 100:+.2f}%")


=== Performance Comparison ===
Baseline Model Accuracy: 100.00%
Tuned Model Accuracy:    100.00%
Absolute Improvement:    +0.00%


### Hyperparameter Tuning using GridSearchCV
Now we will optimize our RandomForest model hyperparameters using the engineered features.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_e, y_train)

best_model = grid_search.best_estimator_
print("Best parameters found:", grid_search.best_params_)

tuned_preds = best_model.predict(X_test_e)
tuned_acc = accuracy_score(y_test, tuned_preds)
print(f"Tuned Model Accuracy: {tuned_acc:.4f}")

Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Tuned Model Accuracy: 1.0000


### Compare Performance
Let's analyze and quantify the improvement between the baseline model and the new tuned model with engineered features.

In [ ]:
improvement = tuned_acc - baseline_acc
print("=== Performance Comparison ===")
print(f"Baseline Model Accuracy: {baseline_acc * 100:.2f}%")
print(f"Tuned Model Accuracy:    {tuned_acc * 100:.2f}%")
print(f"Absolute Improvement:    {improvement * 100:+.2f}%")

=== Performance Comparison ===
Baseline Model Accuracy: 100.00%
Tuned Model Accuracy:    100.00%
Absolute Improvement:    +0.00%
